# Building a RAG Pipeline with LangChain and Azure AI Deployed Models

## Lab Overview

In this lab you will learn how to build a **Retrieval-Augmented Generation (RAG) pipeline using LangChain and Azure-deployed models**.

RAG is a technique that enhances Large Language Model (LLM) responses by:
- **Ingesting** documents and splitting them into chunks
- **Embedding** those chunks into a vector store
- **Retrieving** the most relevant chunks at query time
- **Generating** grounded, context-aware answers using an Azure-deployed LLM

This notebook demonstrates how to connect **Azure OpenAI deployments** (both Chat and Embeddings) via environment variables, build a vector store, create a retrieval chain, and query it with natural language.

## What You Will Learn

By the end of this lab you will be able to:

1. Set up environment variables for Azure OpenAI securely using a `.env` file
2. Connect LangChain to Azure-deployed Chat and Embedding models
3. Load and split documents into manageable chunks
4. Embed document chunks and store them in a FAISS vector store
5. Build a retrieval chain that fetches relevant context
6. Run end-to-end RAG queries and inspect retrieved sources
7. Understand the role of each component in a production RAG pipeline

## Prerequisites

Before starting this lab, make sure you have an active Azure subscription with Azure OpenAI access granted, and a deployed Azure OpenAI resource with two model deployments - one Chat model (e.g., `gpt-4o`) and one Embeddings model (e.g., `text-embedding-ada-002`). You will also need Python 3.9 or higher with the latest version of pip, and a `.env` file placed in the same directory as this notebook (see Step 1 for the exact format).

### Required `.env` file format

Create a file named `.env` in the same folder as this notebook with the following contents:

```
AZURE_OPENAI_API_KEY=your_azure_openai_api_key
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_VERSION=2024-02-01
AZURE_CHAT_DEPLOYMENT_NAME=gpt-4o
AZURE_EMBEDDING_DEPLOYMENT_NAME=text-embedding-ada-002
```

## Step by Step Instructions

### Step 1 - Install Required Libraries

We install all the Python packages needed for this lab. `langchain`, `langchain-openai`, and `faiss-cpu` form the core of our RAG stack, while `python-dotenv` lets us load credentials securely from a `.env` file.

In [1]:
# Install all required dependencies
# Run this cell once - you can skip it on subsequent runs if packages are already installed

%pip install --quiet \
    langchain \
    langchain-openai \
    langchain-community \
    faiss-cpu \
    python-dotenv \
    pypdf \
    tiktoken

print("All packages installed successfully.")

Note: you may need to restart the kernel to use updated packages.
All packages installed successfully.


### Step 2 - Create and Load Environment Variables

We use `python-dotenv` to read credentials from the `.env` file and expose them as environment variables. This keeps API keys out of the notebook code and avoids hardcoding secrets.

In [2]:
from pathlib import Path

env_content = """
AZURE_OPENAI_ENDPOINT=https://<TBD>.cognitiveservices.azure.com
AZURE_OPENAI_API_KEY=
AZURE_CHAT_DEPLOYMENT_NAME=gpt-4o
AZURE_EMBEDDING_DEPLOYMENT_NAME=text-embedding-ada-002
AZURE_OPENAI_API_VERSION=2024-12-01-preview
"""

# Create .env file in current directory
env_path = Path(".env")
env_path.write_text(env_content.strip())

print(".env file created successfully.")
print(env_path.resolve())

.env file created successfully.
\\260302aaibatch.file.core.windows.net\20260302acclaixlabfiles\Day5\Lab1\.env


In [3]:
import os
from dotenv import load_dotenv

# Load all variables defined in .env into the current environment
load_dotenv(override=True)

# Read each variable - we will pass these explicitly to LangChain constructors
AZURE_OPENAI_API_KEY           = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT          = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION       = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_CHAT_DEPLOYMENT_NAME     = os.getenv("AZURE_CHAT_DEPLOYMENT_NAME")
AZURE_EMBEDDING_DEPLOYMENT_NAME = os.getenv("AZURE_EMBEDDING_DEPLOYMENT_NAME")

# Validate that all required variables are present
required_vars = {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_CHAT_DEPLOYMENT_NAME": AZURE_CHAT_DEPLOYMENT_NAME,
    "AZURE_EMBEDDING_DEPLOYMENT_NAME": AZURE_EMBEDDING_DEPLOYMENT_NAME,
}

missing = [k for k, v in required_vars.items() if not v]
if missing:
    raise EnvironmentError(f"Missing environment variables: {missing}. Please check your .env file.")

print("Environment variables loaded successfully.")
print(f"   Endpoint  : {AZURE_OPENAI_ENDPOINT}")
print(f"   Chat model: {AZURE_CHAT_DEPLOYMENT_NAME}")
print(f"   Embeddings: {AZURE_EMBEDDING_DEPLOYMENT_NAME}")

Environment variables loaded successfully.
   Endpoint  : https://<TBD>.cognitiveservices.azure.com
   Chat model: gpt-4o
   Embeddings: text-embedding-ada-002


### Step 3 - Initialize the Azure OpenAI Chat Model

We create a `AzureChatOpenAI` LangChain object pointing at our deployed chat model. This object will later be used by the RAG chain to generate the final answer based on retrieved context.

In [4]:
from langchain_openai import AzureChatOpenAI

# Instantiate the Azure Chat model
# temperature=0 makes the output deterministic - ideal for factual Q&A
llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_CHAT_DEPLOYMENT_NAME,
    openai_api_key=AZURE_OPENAI_API_KEY,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

# Quick sanity-check - invoke the model with a simple prompt
test_response = llm.invoke("Reply with exactly: Azure LLM connection successful.")
print("LLM test:", test_response.content)

LLM test: Azure LLM connection successful.


### Step 4 - Initialize the Azure OpenAI Embeddings Model

We create an `AzureOpenAIEmbeddings` object connected to our Azure-deployed embeddings model. Its job is to transform any piece of text - whether a document chunk or a user query - into a fixed-length list of numbers called a **vector**, where semantically similar texts end up numerically close to each other in that space.

After initializing the model we embed a short test sentence and print the **vector dimension** - the number of numbers in that list. We check this because the dimension is fixed by the model (`text-embedding-ada-002` always produces **1536-dimensional** vectors) and must stay consistent across every document and every query; a mismatch would cause the similarity search to silently fail or produce nonsense rankings.

In [5]:
from langchain_openai import AzureOpenAIEmbeddings

# Instantiate the Azure Embeddings model
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_EMBEDDING_DEPLOYMENT_NAME,
    openai_api_key=AZURE_OPENAI_API_KEY,
    openai_api_version=AZURE_OPENAI_API_VERSION,
)

# Quick sanity-check - embed a short test sentence
test_embedding = embeddings.embed_query("Hello, Azure Embeddings!")
print(f"Embeddings model ready. Vector dimension: {len(test_embedding)}")

Embeddings model ready. Vector dimension: 1536


### Step 5 - Prepare Sample Documents

We define a small set of documents to simulate a knowledge base. In a real scenario you would load PDFs, Word docs, or web pages - LangChain supports all of these through its document loaders.

In [6]:
from langchain_core.documents import Document

# Sample knowledge-base documents
# Each Document has page_content (the text) and metadata (source info for citations)
raw_documents = [
    Document(
        page_content=(
            "Azure OpenAI Service provides REST API access to OpenAI's powerful language models "
            "including GPT-4, GPT-4 Turbo, and GPT-3.5 Turbo. These models can be adapted for "
            "tasks such as content generation, summarization, semantic search, and natural language "
            "to code translation. Users can access the service through REST APIs, Python SDK, or "
            "the Azure OpenAI Studio web-based interface."
        ),
        metadata={"source": "azure_openai_overview.txt", "topic": "Azure OpenAI"},
    ),
    Document(
        page_content=(
            "Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval "
            "with text generation. Instead of relying solely on the parametric knowledge stored in "
            "model weights, RAG systems dynamically retrieve relevant documents from an external "
            "knowledge base and pass them as context to the language model. This reduces hallucinations, "
            "keeps answers grounded in authoritative sources, and allows knowledge to be updated "
            "without retraining the model."
        ),
        metadata={"source": "rag_concept.txt", "topic": "RAG"},
    ),
    Document(
        page_content=(
            "LangChain is an open-source framework designed to simplify the development of applications "
            "powered by large language models. It provides abstractions for chains, agents, memory, "
            "document loaders, text splitters, and vector stores. LangChain integrates natively with "
            "Azure OpenAI through the langchain-openai package, making it easy to swap between "
            "OpenAI-hosted and Azure-hosted model deployments."
        ),
        metadata={"source": "langchain_overview.txt", "topic": "LangChain"},
    ),
    Document(
        page_content=(
            "FAISS (Facebook AI Similarity Search) is an efficient library for similarity search and "
            "clustering of dense vectors. It is widely used as an in memory vector store in RAG "
            "pipelines during prototyping and development. For production workloads, managed vector "
            "databases such as Azure AI Search, Pinecone, or Weaviate are typically preferred because "
            "they offer persistence, scalability, and enterprise security features."
        ),
        metadata={"source": "faiss_overview.txt", "topic": "Vector Stores"},
    ),
    Document(
        page_content=(
            "Azure AI Search (formerly Cognitive Search) is a cloud search service that supports "
            "full-text search, semantic search, and vector search. It integrates with Azure OpenAI "
            "embeddings to enable hybrid retrieval - combining keyword and semantic signals. "
            "Azure AI Search is the recommended vector store for production RAG solutions on Azure "
            "because it supports role-based access control, private endpoints, and auto-scaling."
        ),
        metadata={"source": "azure_ai_search.txt", "topic": "Azure AI Search"},
    ),
]

print(f"Loaded {len(raw_documents)} documents into the knowledge base.")
for doc in raw_documents:
    print(f"    • [{doc.metadata['topic']}] {doc.metadata['source']}")

Loaded 5 documents into the knowledge base.
    • [Azure OpenAI] azure_openai_overview.txt
    • [RAG] rag_concept.txt
    • [LangChain] langchain_overview.txt
    • [Vector Stores] faiss_overview.txt
    • [Azure AI Search] azure_ai_search.txt


### Step 6 - Split Documents into Chunks

Large documents are split into smaller overlapping chunks before embedding. Chunking ensures that each piece fits within the model's context window and that the most relevant passage - rather than an entire document - is retrieved.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size controls the max characters per chunk
# chunk_overlap ensures continuity between adjacent chunks (avoids cutting sentences mid-thought)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    length_function=len,
    add_start_index=True,  # adds char offset metadata for traceability
)

chunks = text_splitter.split_documents(raw_documents)

print(f"Split {len(raw_documents)} documents into {len(chunks)} chunks.")
print("\nSample chunk:")
print(f"Content : {chunks[0].page_content[:200]}...")
print(f"Metadata: {chunks[0].metadata}")

Split 5 documents into 5 chunks.

Sample chunk:
Content : Azure OpenAI Service provides REST API access to OpenAI's powerful language models including GPT-4, GPT-4 Turbo, and GPT-3.5 Turbo. These models can be adapted for tasks such as content generation, su...
Metadata: {'source': 'azure_openai_overview.txt', 'topic': 'Azure OpenAI', 'start_index': 0}


### Step 7 - Create a FAISS Vector Store

We embed all document chunks using our Azure embeddings model and store the resulting vectors in a FAISS index. FAISS enables fast nearest-neighbor search so we can retrieve the most semantically relevant chunks for any given query.

In [8]:
from langchain_community.vectorstores import FAISS

print("Embedding document chunks and building FAISS index...")

# FAISS.from_documents calls embeddings.embed_documents() on all chunks
# and stores (vector, document) pairs in an in-memory index
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print(f"FAISS vector store created with {vector_store.index.ntotal} vectors.")

# --- Optional: persist to disk so you don't re-embed on every run ---
# vector_store.save_local("faiss_index")
# vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

# print(f"FAISS vector store stored locally.")

Embedding document chunks and building FAISS index...
FAISS vector store created with 5 vectors.


### Step 8 - Create a Retriever

A `retriever` is a LangChain interface that accepts a query string and returns the top-k most relevant document chunks. Wrapping the vector store as a retriever lets us plug it seamlessly into a RAG chain.

In [9]:
# k=3 means we retrieve the 3 most similar chunks for each query
# Increasing k gives more context but raises the risk of irrelevant passages
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# Test the retriever independently before wiring it into the full chain
test_query = "What is RAG and why is it useful?"
retrieved_docs = retriever.invoke(test_query)

print(f"Retriever test - query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} chunks:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"   [{i}] Source: {doc.metadata.get('source')}")
    print(f"       Preview: {doc.page_content[:120]}...\n")

Retriever test - query: 'What is RAG and why is it useful?'
Retrieved 3 chunks:

   [1] Source: rag_concept.txt
       Preview: Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of...

   [2] Source: faiss_overview.txt
       Preview: FAISS (Facebook AI Similarity Search) is an efficient library for similarity search and clustering of dense vectors. It ...

   [3] Source: azure_ai_search.txt
       Preview: Azure AI Search (formerly Cognitive Search) is a cloud search service that supports full-text search, semantic search, a...



### Step 9 - Define the RAG Prompt Template

We define a structured prompt that instructs the LLM to answer **only** using the retrieved context. This grounding prevents hallucination and keeps the model focused on the authoritative knowledge base we have built.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

# The prompt has two dynamic placeholders:
#   {context} - injected retrieved chunks
#   {question} - the user's original query
RAG_PROMPT_TEMPLATE = """\
You are a helpful AI assistant. Answer the question using only the context provided below.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

print("RAG prompt template defined.")
print("Placeholders:", prompt.input_variables)

RAG prompt template defined.
Placeholders: ['context', 'question']


### Step 10 - Build the RAG Chain

We compose the full RAG pipeline using LangChain Expression Language (LCEL). The chain runs in sequence: retrieve -> format context -> fill prompt -> call LLM -> parse output.

In [11]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    """Concatenate retrieved chunks into a single context string."""
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# LCEL pipeline - each | operator pipes the output of one step into the next
rag_chain = (
    {
        "context":  retriever | format_docs,   # Step A: retrieve & format docs
        "question": RunnablePassthrough(),     # Step B: pass the question through unchanged
    }
    | prompt       # Step C: inject context + question into the prompt template
    | llm          # Step D: call the Azure Chat LLM
    | StrOutputParser()  # Step E: extract the plain-text response
)

print("RAG chain assembled and ready.")

RAG chain assembled and ready.


### Step 11 - Run RAG Queries

We invoke the chain with several questions to see end-to-end RAG in action. Each call triggers retrieval, context injection, and LLM generation - producing grounded answers from the knowledge base.

In [12]:
def ask(question: str) -> str:
    """Run a question through the RAG chain and print a formatted response."""
    print(f"\n{'='*65}")
    print(f"Question: {question}")
    print("="*65)
    answer = rag_chain.invoke(question)
    print(f"Answer:\n{answer}")
    return answer

ask("What is Langchain?")


Question: What is Langchain?
Answer:
LangChain is an open-source framework designed to simplify the development of applications powered by large language models. It provides abstractions for chains, agents, memory, document loaders, text splitters, and vector stores. LangChain integrates natively with Azure OpenAI through the langchain-openai package, making it easy to swap between OpenAI-hosted and Azure-hosted model deployments.


'LangChain is an open-source framework designed to simplify the development of applications powered by large language models. It provides abstractions for chains, agents, memory, document loaders, text splitters, and vector stores. LangChain integrates natively with Azure OpenAI through the langchain-openai package, making it easy to swap between OpenAI-hosted and Azure-hosted model deployments.'

### Step 12 - Inspect Retrieved Sources (Transparency)

For production applications it is important to show users *which* sources were used to generate an answer. Here we build a version of the chain that returns both the answer and the source documents for full transparency.

In [13]:
from langchain_core.runnables import RunnableParallel

# We run the retriever twice - once to feed the LLM, once to return sources to the caller
rag_chain_with_sources = RunnableParallel(
    {
        "answer": rag_chain,
        "sources": retriever,
    }
)

# Run a query and display both the answer and its source documents
query = "What vector stores are recommended for production RAG on Azure?"
result = rag_chain_with_sources.invoke(query)

print(f"\n{'='*65}")
print(f"Question: {query}")
print("="*65)
print(f"\nAnswer:\n{result['answer']}")

print(f"\nSources used ({len(result['sources'])} chunks):")
for i, src in enumerate(result["sources"], 1):
    print(f"   [{i}] {src.metadata.get('source')} (topic: {src.metadata.get('topic')})")
    print(f"       {src.page_content[:100]}...")


Question: What vector stores are recommended for production RAG on Azure?

Answer:
Azure AI Search is the recommended vector store for production RAG solutions on Azure.

Sources used (3 chunks):
   [1] azure_ai_search.txt (topic: Azure AI Search)
       Azure AI Search (formerly Cognitive Search) is a cloud search service that supports full-text search...
   [2] rag_concept.txt (topic: RAG)
       Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text ge...
   [3] faiss_overview.txt (topic: Vector Stores)
       FAISS (Facebook AI Similarity Search) is an efficient library for similarity search and clustering o...


### Step 13 - (Optional) Load Real Documents

In real projects you will ingest documents from files or URLs rather than typed strings. LangChain provides ready-made loaders for PDFs, Word docs, web pages, and more - the rest of the pipeline stays exactly the same.

In [14]:
# ─── Example A: Load a PDF file ──────────────────────────────────────────────
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("/data/langchain_azure_openai.pdf")
# pdf_docs = loader.load()

# ─── Example B: Load a web page ───────────────────────────────────────────────
# from langchain_community.document_loaders import WebBaseLoader
# loader = WebBaseLoader("https://learn.microsoft.com/en-us/azure/foundry/concepts/foundry-models-overview")
# web_docs = loader.load()

# ─── Example C: Load a plain-text file ───────────────────────────────────────
from langchain_community.document_loaders import TextLoader
loader = TextLoader("./data/knowledge_base.txt", encoding="utf-8")
text_docs = loader.load()

# After loading, feed documents through the same pipeline:
# chunks = text_splitter.split_documents(loaded_docs)
chunks = text_splitter.split_documents(text_docs)
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

#### Interactive chat loop over your loaded documents
Once your retriever is rebuilt above, run this cell to chat with your docs.
#### Type your question and press Enter. Type 'exit' to stop.

Example prompt: How can we connect langchain and azure openai?

In [15]:
print("\nChat with your documents (type 'exit' to quit)\n")
while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() == "exit":
        print("Ending chat session.")
        break
    answer = rag_chain.invoke(user_input)
    print(f"\nAssistant: {answer}\n")


Chat with your documents (type 'exit' to quit)


Assistant: LangChain is an open-source framework designed to simplify the development of applications powered by large language models. It provides abstractions for chains, agents, memory, document loaders, text splitters, and vector stores. LangChain integrates natively with Azure OpenAI through the langchain-openai package, making it easy to swap between OpenAI-hosted and Azure-hosted model deployments.


Assistant: LangChain integrates natively with Azure OpenAI through the langchain-openai package, making it easy to connect and swap between OpenAI-hosted and Azure-hosted model deployments.


Assistant: Azure OpenAI Service provides REST API access to OpenAI's powerful language models, including GPT-4, GPT-4 Turbo, and GPT-3.5 Turbo. These models can be used for tasks such as content generation, summarization, semantic search, and natural language to code translation. Users can access the service through REST APIs, Python SDK, or the

## Summary

Congratulations on completing the lab! You built a full RAG pipeline from scratch - loading credentials securely from a `.env` file, connecting to Azure-deployed chat and embeddings models, chunking and indexing documents into a FAISS vector store, and wiring everything together into a LangChain LCEL chain that retrieves relevant context before generating grounded answers.

The key idea behind RAG is simple: instead of asking the LLM to rely on what it memorized during training, you hand it the right information at query time. This reduces hallucinations, keeps answers up to date, and makes the system auditable since you can always trace which source chunks were used.
